# test_v1 - 자유형 8단계 동화 계획 엔진 + 통합 AI 기능 테스트

- Qwen3.8-27B GGUF로 8단계 동화 계획, 장면, 선택지를 생성합니다.
- 감정은 별도 체크포인트 없이 Qwen이 동화 문맥과 선택을 읽어 서사적 정서 점수로 해설합니다.
- Google Drive를 마운트하지 않으며, 모델 캐시는 Colab 런타임 종료 시 사라집니다.


In [ ]:
# 셀 1: 패키지 설치
# Google Drive와 별도 감정 분류 모델을 사용하지 않는 구성입니다.
# Colab 임시 저장소에 모델을 내려받으므로 런타임이 종료되면 다음 실행 때 다시 다운로드됩니다.
import os
import shutil
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf_transfer"])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers==4.43.0", "fastapi", "uvicorn[standard]", "python-multipart", "accelerate", "nest-asyncio",
    "hf_transfer", "requests", "pillow", "virtualenv", "faster-whisper>=1.1.0",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "diffusers", "invisible_watermark", "safetensors", "sentencepiece",
])

# Qwen3.8 GGUF는 최신 llama.cpp CUDA 빌드가 필요합니다.
cuda_wheel_index = "https://abetlen.github.io/llama-cpp-python/whl/cu124"
try:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "-U", "--force-reinstall", "--no-cache-dir",
        "llama-cpp-python", "--extra-index-url", cuda_wheel_index,
    ])
except subprocess.CalledProcessError:
    os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on"
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "-U", "--force-reinstall", "--no-cache-dir",
        "--no-binary", "llama-cpp-python", "llama-cpp-python",
    ])

# XTTS 의존성은 메인 Qwen 환경과 분리해 설치합니다.
XTTS_VENV = "/content/test_v1_xtts_env"
XTTS_PYTHON = f"{XTTS_VENV}/bin/python"
if not os.path.isfile(XTTS_PYTHON):
    shutil.rmtree(XTTS_VENV, ignore_errors=True)
    subprocess.check_call([sys.executable, "-m", "virtualenv", "--system-site-packages", XTTS_VENV])
subprocess.check_call([XTTS_PYTHON, "-m", "pip", "install", "-q", "-U", "pip"])
subprocess.check_call([
    XTTS_PYTHON, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall",
    "coqui-tts==0.27.5", "transformers==4.57.6", "fastapi", "uvicorn[standard]", "python-multipart",
])

# Flutter와 연결할 공개 URL용 Cloudflare 터널입니다.
if not shutil.which("cloudflared"):
    cloudflared_deb = "/content/cloudflared.deb"
    subprocess.check_call([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
        "-O", cloudflared_deb,
    ])
    subprocess.check_call(["dpkg", "-i", cloudflared_deb])

print("설치 완료. Drive 마운트 없이 셀 2부터 실행하세요.")


In [ ]:
# 셀 2: Drive 없이 Qwen, DreamShaper/XTTS 설정을 준비합니다.
import gc
import json
import os
from pathlib import Path

import torch

# Colab 런타임의 임시 저장소만 사용합니다. 런타임 종료 후 캐시는 사라집니다.
HF_CACHE_ROOT = "/content/fairytale_hf_cache"
os.environ["HF_HOME"] = f"{HF_CACHE_ROOT}/home"
os.environ["HF_HUB_CACHE"] = f"{HF_CACHE_ROOT}/hub"
os.environ["TTS_HOME"] = "/content/fairytale_xtts_cache"
STT_CACHE_ROOT = "/content/fairytale_stt_cache"
STT_MODEL_NAME = "large-v3-turbo"
for directory in [os.environ["HF_HOME"], os.environ["HF_HUB_CACHE"], os.environ["TTS_HOME"], STT_CACHE_ROOT]:
    Path(directory).mkdir(parents=True, exist_ok=True)

from huggingface_hub import login
from llama_cpp import Llama

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)

MODEL_REPO = "unsloth/Qwen3.8-27B-GGUF"
MODEL_FILE = "Qwen3.8-27B-UD-Q4_K_M.gguf"
N_CTX = 8192
N_BATCH = 256
XTTS_ACCEPT_LICENSE = True

# 동화 문맥을 읽는 Qwen이 감정 이름과 점수를 직접 생성합니다.
LLM_EMOTION_LABELS = [
    "기쁨", "기대감", "호기심", "용기", "친절함", "안심",
    "놀람", "걱정", "슬픔", "화남", "감동", "즐거움",
]
EMOTION_LABEL_SOURCE = "qwen_contextual_analysis"

if not torch.cuda.is_available():
    raise RuntimeError("GPU 런타임이 필요합니다. Colab 런타임 유형을 GPU로 바꾸세요.")
GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {torch.cuda.get_device_name(0)} / VRAM {GPU_VRAM_GB:.1f}GB")
if GPU_VRAM_GB < 22:
    raise RuntimeError("Qwen3.8-27B Q4에는 최소 22GB 정도의 GPU 메모리가 필요합니다. L4/A100 런타임을 사용하세요.")

HIGH_VRAM_MODE = GPU_VRAM_GB >= 36
IMAGE_DEVICE = "cuda" if HIGH_VRAM_MODE else "cpu"
XTTS_DEVICE = "cuda" if HIGH_VRAM_MODE else "cpu"
STT_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
STT_COMPUTE_TYPE = "float16" if STT_DEVICE == "cuda" else "int8"
os.environ["XTTS_ACCEPT_LICENSE"] = str(XTTS_ACCEPT_LICENSE).lower()
os.environ["COQUI_TOS_AGREED"] = "1" if XTTS_ACCEPT_LICENSE else ""
os.environ["XTTS_DEVICE"] = XTTS_DEVICE

print("Qwen 로딩 중...")
llm = Llama.from_pretrained(
    repo_id=MODEL_REPO,
    filename=MODEL_FILE,
    n_ctx=N_CTX,
    n_batch=N_BATCH,
    n_gpu_layers=-1,
    flash_attn=True,
    verbose=False,
)
print("Qwen 준비 완료")
print("감정 분석: Qwen 문맥 분석 방식")
print(f"삽화 장치: {IMAGE_DEVICE}, XTTS 장치: {XTTS_DEVICE}")
print(f"음성 인식: faster-whisper {STT_MODEL_NAME} ({STT_DEVICE}/{STT_COMPUTE_TYPE}, 첫 사용 시 로드)")


In [ ]:
# 셀 3: Qwen 호출, 느슨한 JSON 파싱, 텍스트 정리 유틸리티
import ast
import html
import re
import unicodedata


def qwen_prompt(user_prompt):
    # 생각 블록을 닫아 둬서 사용자 화면에 추론 태그가 섞이지 않게 합니다.
    return (
        "<|im_start|>user\n" + user_prompt.strip() +
        "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
    )


def call_llm(prompt, max_tokens, temperature=0.55, top_p=0.9):
    result = llm.create_completion(
        prompt=qwen_prompt(prompt),
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        repeat_penalty=1.1,
        stop=["<|im_end|>", "<|im_start|>"],
    )
    return str(result["choices"][0]["text"]).strip()


def clean_model_text(value):
    text = html.unescape(str(value or ""))
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    text = re.sub(r"```(?:json)?", "", text, flags=re.I)
    text = text.replace("</CHOICES>", "").replace("</STORY>", "")
    text = re.sub(r"(?m)^\s*(?:Human|Assistant)\s*:\s*", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def strip_json_comments(text):
    output, index, quoted, escaped = [], 0, False, False
    while index < len(text):
        char = text[index]
        nxt = text[index + 1] if index + 1 < len(text) else ""
        if quoted:
            output.append(char)
            if escaped:
                escaped = False
            elif char == "\\":
                escaped = True
            elif char == '"':
                quoted = False
            index += 1
        elif char == '"':
            quoted = True
            output.append(char)
            index += 1
        elif char == "/" and nxt == "/":
            newline = text.find("\n", index)
            index = len(text) if newline < 0 else newline + 1
        elif char == "/" and nxt == "*":
            close = text.find("*/", index + 2)
            index = len(text) if close < 0 else close + 2
        else:
            output.append(char)
            index += 1
    return "".join(output)


def parse_json_loose(fragment):
    if not isinstance(fragment, str):
        return fragment if isinstance(fragment, (dict, list)) else None
    repaired = strip_json_comments(fragment)
    repaired = repaired.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")
    repaired = re.sub(r",\s*([}\]])", r"\1", repaired)
    for candidate in (fragment, repaired):
        try:
            return json.loads(candidate)
        except Exception:
            try:
                value = ast.literal_eval(candidate)
                if isinstance(value, (dict, list)):
                    return value
            except Exception:
                pass
    return None


def extract_json_loose(text):
    # 설명문 안의 대괄호보다 실제 JSON 객체/배열을 우선 안전하게 찾습니다.
    text = clean_model_text(text)
    pairs = {"{": "}", "[": "]"}
    for start, opener in enumerate(text):
        if opener not in pairs:
            continue
        stack, quoted, escaped = [pairs[opener]], False, False
        for position in range(start + 1, len(text)):
            char = text[position]
            if quoted:
                if escaped:
                    escaped = False
                elif char == "\\":
                    escaped = True
                elif char == '"':
                    quoted = False
                continue
            if char == '"':
                quoted = True
            elif char in pairs:
                stack.append(pairs[char])
            elif char in "}]":
                if not stack or char != stack[-1]:
                    break
                stack.pop()
                if not stack:
                    parsed = parse_json_loose(text[start:position + 1])
                    if isinstance(parsed, (dict, list)):
                        return parsed
                    break
    return None


def sentences(text):
    return [part.strip() for part in re.split(r"(?<=[.!?…])\s+|\n+", clean_model_text(text)) if part.strip()]


def clean_story(text, headings=()):
    text = clean_model_text(text)
    text = re.sub(r"(?m)^\s*(?:제목|장면|본문|단계)\s*:\s*[^\n]*$", "", text)
    text = re.sub(r"(?m)^\s*#{1,6}\s+.*$", "", text)
    text = text.replace("**", "").replace("__", "")
    text = "".join(char for char in text if unicodedata.category(char) not in {"So", "Sk"})
    normalized_headings = {
        re.sub(r"\s+", "", clean_model_text(heading))
        for heading in headings if clean_model_text(heading)
    }
    kept_lines = []
    for line in text.splitlines():
        compact = re.sub(r"\s+", "", line.strip())
        if compact and compact in normalized_headings:
            continue
        kept_lines.append(line)
    text = "\n".join(kept_lines)
    text = re.sub(r"\s+([,.!?])", r"\1", text)
    text = re.sub(r"([!?])\1+", r"\1", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def compact_json(data, limit=5200):
    result = json.dumps(data, ensure_ascii=False, separators=(",", ":"))
    return result if len(result) <= limit else result[:limit] + "..."


# ── Qwen 문맥 감정 분석 ─────────────────────────────────────────────────────
def _emotion_item(label, score):
    index = LLM_EMOTION_LABELS.index(label)
    return {
        "label_index": index,
        "label": label,
        "label_display": label,
        "score": round(float(score), 4),
    }


def _emotion_fallback():
    return [
        _emotion_item("호기심", 0.72),
        _emotion_item("기대감", 0.66),
        _emotion_item("용기", 0.58),
    ]


def predict_emotions(text, top_k=5, threshold=0.25):
    labels = ", ".join(LLM_EMOTION_LABELS)
    prompt = f'''너는 어린이 동화의 장면과 선택에서 드러나는 정서를 읽는 해설가다. 사람의 실제 성격이나 정신 상태를 진단하지 않는다.

[분석할 텍스트]
{clean_model_text(text)[-2600:]}

아래 감정 이름만 사용한다: {labels}
텍스트에 실제로 드러난 분위기를 기준으로 서로 다른 감정 3~5개와 0~1 점수를 정한다. 점수는 절대적인 심리검사가 아니라 이번 장면의 서사적 정서 강도다.
JSON만 출력한다.
{{"primary_emotion":"감정 이름","primary_score":0.0,"top_emotions":[{{"label":"감정 이름","score":0.0}}]}}'''
    raw = call_llm(prompt, max_tokens=360, temperature=0.15, top_p=0.8)
    data = extract_json_loose(raw)
    raw_items = data.get("top_emotions", []) if isinstance(data, dict) else []
    pairs, seen = [], set()
    for item in raw_items if isinstance(raw_items, list) else []:
        if not isinstance(item, dict):
            continue
        label = clean_model_text(item.get("label", ""))
        if label not in LLM_EMOTION_LABELS or label in seen:
            continue
        try:
            score = round(max(0.0, min(1.0, float(item.get("score", 0)))), 4)
        except Exception:
            continue
        seen.add(label)
        pairs.append(_emotion_item(label, score))
    if len(pairs) < 3:
        for item in _emotion_fallback():
            if item["label"] not in seen:
                pairs.append(item)
                seen.add(item["label"])
            if len(pairs) >= 3:
                break
    pairs.sort(key=lambda item: item["score"], reverse=True)
    pairs = pairs[:top_k]
    primary = clean_model_text(data.get("primary_emotion", "")) if isinstance(data, dict) else ""
    if primary not in {item["label"] for item in pairs}:
        primary = pairs[0]["label"]
    primary_score = next(item["score"] for item in pairs if item["label"] == primary)
    active = [item for item in pairs if item["score"] >= threshold] or pairs[:1]
    return {
        "primary_emotion": primary,
        "primary_score": primary_score,
        "top_emotions": pairs,
        "active_emotions": active,
        "scores": {item["label"]: item["score"] for item in pairs},
        "scores_by_index": {str(item["label_index"]): item["score"] for item in pairs},
        "emotion_label_source": EMOTION_LABEL_SOURCE,
        "emotion_labels_are_generic": False,
        "primary_emotion_index": LLM_EMOTION_LABELS.index(primary),
        "primary_emotion_display": primary,
        "emotion_disclaimer": "Qwen이 이번 동화 장면의 서사적 정서를 해설한 결과입니다.",
    }


def analyze_choice_emotion(story_so_far, choice_text):
    result = predict_emotions(f"[앞 장면]\n{story_so_far[-1800:]}\n\n[선택]\n{choice_text}")
    result["mode"] = "choice"
    result["choice"] = choice_text
    return result


def analyze_story_emotion(story_text):
    result = predict_emotions(story_text[-2200:])
    result["mode"] = "story"
    return result


# ── DreamShaper 8 삽화: Qwen VRAM을 위해 필요할 때만 로드합니다. ────────────
image_pipe = None


def get_image_pipe():
    global image_pipe
    if image_pipe is not None:
        return image_pipe
    from diffusers import DPMSolverMultistepScheduler, StableDiffusionPipeline
    dtype = torch.float16 if IMAGE_DEVICE == "cuda" else torch.float32
    print(f"DreamShaper 8 로딩 중... ({IMAGE_DEVICE})")
    pipe = StableDiffusionPipeline.from_pretrained(
        "Lykon/dreamshaper-8",
        torch_dtype=dtype,
        safety_checker=None,
        requires_safety_checker=False,
    )
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, algorithm_type="dpmsolver++")
    image_pipe = pipe.to(IMAGE_DEVICE)
    image_pipe.enable_attention_slicing()
    return image_pipe


def generate_illustration(story_text, genre="판타지"):
    style = {
        "판타지": "magical Korean children picture book, soft watercolors, glowing forest",
        "모험": "Korean children picture book, adventurous landscape, warm sunlight",
        "우정": "Korean children picture book, warm friendship, gentle colors",
    }.get(genre, "Korean children picture book, warm detailed illustration")
    visual = call_llm(
        "다음 동화 장면을 삽화 프롬프트용 영어 한 문장으로 바꿔라. 인물, 장소, 행동만 쓰고 22단어 이내로 쓴다.\n\n" + story_text[-1100:],
        max_tokens=70,
        temperature=0.2,
    )
    visual = re.sub(r"[가-힣]+", "", visual).strip() or "a child follows a glowing path through an enchanted forest"
    result = get_image_pipe()(
        prompt=f"{visual}, {style}, high quality illustration, no text",
        negative_prompt="scary, violent, dark horror, text, watermark, blurry, realistic photo",
        num_inference_steps=20,
        guidance_scale=7.5,
        width=512,
        height=512,
    )
    return result.images[0]


# ── XTTS-v2: 내 목소리 WAV가 있을 때만 모델을 지연 로드합니다. ──────────────
import base64
import subprocess
import time
import requests

XTTS_APP_PATH = Path("/content/test_v1_xtts_sidecar.py")
XTTS_LOG_PATH = Path("/content/test_v1_xtts_sidecar.log")
xtts_process = None

XTTS_APP_SOURCE = r"""
import base64, os, re, threading, wave
from io import BytesIO
from pathlib import Path
from tempfile import TemporaryDirectory

import torch
from fastapi import FastAPI, HTTPException
from fastapi.responses import Response
from pydantic import BaseModel
from TTS.api import TTS

app = FastAPI()
device = os.getenv('XTTS_DEVICE', 'cpu')
accepted = os.getenv('XTTS_ACCEPT_LICENSE', '').lower() == 'true'
model = None
lock = threading.Lock()

class RequestBody(BaseModel):
    text: str
    speaker_wav_b64: str

def get_model():
    global model
    if not accepted:
        raise RuntimeError('XTTS-v2 라이선스 확인값이 필요합니다.')
    if model is None:
        model = TTS(model_name='tts_models/multilingual/multi-dataset/xtts_v2', progress_bar=True).to(device)
    return model

@app.get('/health')
def health():
    return {'status':'ok', 'loaded': model is not None, 'device': device}

@app.post('/synthesize')
def synthesize(body: RequestBody):
    if not body.text.strip():
        raise HTTPException(400, '읽을 텍스트가 비어 있습니다.')
    try:
        wav = base64.b64decode(body.speaker_wav_b64, validate=True)
        if wav[:4] != b'RIFF' or wav[8:12] != b'WAVE':
            raise ValueError('PCM WAV 파일이 필요합니다.')
        with lock, TemporaryDirectory() as temp:
            speaker = Path(temp) / 'speaker.wav'
            output = Path(temp) / 'output.wav'
            speaker.write_bytes(wav)
            get_model().tts_to_file(text=body.text[:2500], speaker_wav=str(speaker), language='ko', file_path=str(output), split_sentences=True)
            return Response(output.read_bytes(), media_type='audio/wav')
    except ValueError as error:
        raise HTTPException(400, str(error))
    except Exception as error:
        raise HTTPException(503, str(error))
"""


def ensure_xtts_sidecar():
    global xtts_process
    if xtts_process is not None and xtts_process.poll() is None:
        return
    XTTS_APP_PATH.write_text(XTTS_APP_SOURCE, encoding="utf-8")
    log_file = XTTS_LOG_PATH.open("w")
    xtts_process = subprocess.Popen(
        [XTTS_PYTHON, "-m", "uvicorn", "test_v1_xtts_sidecar:app", "--host", "127.0.0.1", "--port", "8002"],
        cwd="/content",
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )
    for _ in range(30):
        try:
            if requests.get("http://127.0.0.1:8002/health", timeout=2).ok:
                return
        except Exception:
            time.sleep(1)
    details = XTTS_LOG_PATH.read_text(errors="replace")[-3000:]
    raise RuntimeError("XTTS 워커 시작 실패:\n" + details)


def synthesize_with_my_voice(text, speaker_wav_path):
    speaker_path = Path(speaker_wav_path)
    if not speaker_path.is_file():
        raise FileNotFoundError(f"화자 WAV 파일이 없습니다: {speaker_path}")
    ensure_xtts_sidecar()
    response = requests.post(
        "http://127.0.0.1:8002/synthesize",
        json={"text": text, "speaker_wav_b64": base64.b64encode(speaker_path.read_bytes()).decode("ascii")},
        timeout=900,
    )
    response.raise_for_status()
    return response.content


In [ ]:
# 셀 4: 자유형 8단계 계획 생성기
# 인물의 역할 종류와 수를 코드에서 제한하지 않습니다. 모델이 이야기마다 cast를 설계합니다.
STAGE_ARCS = [
    "사건의 발생과 주인공의 목표 설정",
    "목표로 향할 수 있는 방향 또는 의미 있는 단서 확보",
    "관계·정보·도구 중 필요한 준비를 갖춤",
    "되돌아가기 어려운 실제 여정을 시작함",
    "목표를 어렵게 만드는 구체적인 시험을 겪음",
    "갈등의 진실 또는 해결 실마리를 발견하고 마지막 접근을 준비함",
    "가장 큰 갈등을 지혜롭고 안전하게 해결하여 목표를 이룸",
    "사건 이후 달라진 관계와 일상을 보여 주며 완결함",
]


def normalize_cast(raw_cast):
    if isinstance(raw_cast, dict):
        raw_cast = [
            {"name": key, "description": value, "enters_at": 1}
            for key, value in raw_cast.items()
        ]
    if not isinstance(raw_cast, list):
        return []
    cast, seen = [], set()
    for item in raw_cast:
        if isinstance(item, str):
            item = {"name": item, "description": "이야기에서 중요한 역할을 한다.", "enters_at": 1}
        if not isinstance(item, dict):
            continue
        name = clean_model_text(item.get("name", ""))
        if len(name) < 1 or name in seen:
            continue
        seen.add(name)
        try:
            enters_at = int(item.get("enters_at", 1))
        except Exception:
            enters_at = 1
        cast.append({
            "name": name,
            "description": clean_model_text(item.get("description", item.get("role", ""))) or "이야기의 중요한 인물이다.",
            "desire": clean_model_text(item.get("desire", "")),
            "enters_at": max(1, min(8, enters_at)),
        })
    return cast


def normalize_stages(raw_stages):
    if not isinstance(raw_stages, list):
        return []
    stages = []
    for index, item in enumerate(raw_stages[:8], 1):
        if isinstance(item, list):
            item = {
                "title": item[0] if len(item) > 0 else "",
                "setting": item[1] if len(item) > 1 else "",
                "goal": item[2] if len(item) > 2 else "",
                "conflict": item[3] if len(item) > 3 else "",
                "progress": item[4] if len(item) > 4 else "",
            }
        if not isinstance(item, dict):
            continue
        stage = {
            "number": index,
            "arc": STAGE_ARCS[index - 1],
            "title": clean_model_text(item.get("title", "")) or f"{index}번째 장면",
            "setting": clean_model_text(item.get("setting", item.get("location", ""))),
            "goal": clean_model_text(item.get("goal", item.get("required_event", ""))),
            "question": clean_model_text(item.get("question", item.get("dramatic_question", ""))),
            "conflict": clean_model_text(item.get("conflict", item.get("obstacle", ""))),
            "progress": clean_model_text(item.get("progress", item.get("stage_result", item.get("result", "")))),
            "not_yet": clean_model_text(item.get("not_yet", item.get("must_not_happen", ""))),
        }
        stages.append(stage)
    return stages


def normalize_plan(candidate, request):
    if not isinstance(candidate, dict):
        return None
    stages = normalize_stages(candidate.get("stages"))
    cast = normalize_cast(candidate.get("cast", candidate.get("characters", [])))
    # 내용 품질 점수로 생성 자체를 막지 않습니다. 8단계 구조와 최소 인물 목록만 확인합니다.
    if len(stages) != 8 or not cast:
        return None
    world_rules = candidate.get("world_rules", candidate.get("story_bible", []))
    if isinstance(world_rules, str):
        world_rules = [world_rules]
    if not isinstance(world_rules, list):
        world_rules = []
    threads = candidate.get("threads", candidate.get("plot_threads", []))
    if isinstance(threads, str):
        threads = [threads]
    if not isinstance(threads, list):
        threads = []
    return {
        "title": clean_model_text(candidate.get("title", "")) or "이름 없는 모험",
        "request": request,
        "lesson": clean_model_text(candidate.get("lesson", "")),
        "cast": cast,
        "world_rules": [clean_model_text(item) for item in world_rules if clean_model_text(item)],
        "threads": [clean_model_text(item) for item in threads if clean_model_text(item)],
        "stages": stages,
    }


def repair_plan_format(raw, request):
    prompt = f"""아래는 이미 작성된 동화 계획이다. 사건, 인물, 이름, 순서를 새로 만들거나 바꾸지 말고 JSON 형식만 고쳐라.

[원문]
{raw[:12000]}

반드시 아래 키를 가진 JSON 객체 하나만 출력한다.
{{
  "title":"제목",
  "lesson":"교훈",
  "cast":[{{"name":"인물 또는 중요한 존재","description":"성격과 이야기 속 기능","desire":"원하는 것","enters_at":1}}],
  "world_rules":["세계관 또는 마법 규칙"],
  "threads":["끝까지 이어질 단서 또는 관계"],
  "stages":[{{"title":"장면 제목","setting":"장소","goal":"이번 단계에서 반드시 진전될 일","question":"독자가 궁금해할 것","conflict":"장애물","progress":"장면 뒤 확정 사실","not_yet":"아직 해결하면 안 되는 것"}}]
}}
stages는 반드시 원문의 순서를 유지한 8개다. JSON 밖의 말은 쓰지 않는다."""
    repaired = call_llm(prompt, max_tokens=2600, temperature=0.05, top_p=0.8)
    return normalize_plan(extract_json_loose(repaired), request)


def make_story_plan(request, reader_age="초등학교 1~3학년"):
    stage_lines = "\n".join(f"{index + 1}. {arc}" for index, arc in enumerate(STAGE_ARCS))
    prompt = f"""너는 한국어 창작동화의 기획자다. 아래 요청을 바탕으로 처음부터 끝까지 이어지는 8단계 인터랙티브 동화 계획을 만든다.

독자: {reader_age}
사용자 요청: {request}

8단계의 서사적 기능:
{stage_lines}

중요한 자유도 규칙:
- 인물 수, 이름, 종족, 관계, 핵심 물건의 유무는 이야기 필요에 따라 자유롭게 결정한다.
- 주인공·동료·안내자·악당 같은 고정 역할 칸을 억지로 채우지 않는다. 필요한 존재만 만든다.
- 각 인물은 name, description, desire, enters_at(처음 본격적으로 등장하는 1~8 단계)를 가진다.
- 갈등 상대가 있다면 단순히 나쁜 존재로 만들지 말고 납득할 수 있는 바람이나 오해를 준다. 갈등 상대가 필요 없으면 만들지 않아도 된다.
- 세계관과 마법에는 1~4개의 구체적인 규칙을 정한다. 무엇이든 해결하는 마법은 금지한다.
- threads에는 이야기 끝까지 회수할 단서, 약속, 물건, 관계를 1~3개 넣는다.
- 1~6단계에서 최종 문제를 해결하지 않는다. 7단계에서 목표를 이루고, 8단계는 새로운 위기 없이 따뜻하게 끝낸다.
- 앞 단계 progress가 다음 단계 goal의 원인이 되게 한다. 인물과 물건은 enters_at 이전에 실제 행동을 하지 않는다.
- 선택에 따라 장면의 방법과 대화는 달라질 수 있지만, 8단계의 큰 사건 흐름은 유지되어야 한다.
- 어린이가 이해할 수 있는 자연스러운 한국어를 사용하고 게임 퀘스트, 미션, 레벨 같은 표현은 쓰지 않는다.
- JSON 하나만 출력한다. 코드블록, 주석, 설명, 끝 쉼표는 절대 쓰지 않는다.

JSON 형식:
{{
  "title":"구체적이고 동화다운 제목",
  "lesson":"이야기 속 행동으로 드러나는 교훈",
  "cast":[{{"name":"이름","description":"성격과 이야기 속 기능","desire":"바라는 것","enters_at":1}}],
  "world_rules":["세계관 또는 마법 규칙"],
  "threads":["회수할 단서 또는 관계"],
  "stages":[
    {{"title":"장면 제목","setting":"장소","goal":"이번 단계에서 반드시 진전될 일","question":"독자가 궁금해할 것","conflict":"구체적 장애물","progress":"장면 뒤 확정 사실","not_yet":"아직 해결하면 안 되는 일"}}
  ]
}}
stages는 정확히 8개를 써야 한다."""
    raw = call_llm(prompt, max_tokens=3600, temperature=0.38, top_p=0.88)
    plan = normalize_plan(extract_json_loose(raw), request)
    if plan is None:
        print("[정보] 계획 내용은 유지한 채 JSON 형식만 한 번 복구합니다.")
        plan = repair_plan_format(raw, request)
    if plan is None:
        print("[계획 생성 원문]\n" + raw[:12000])
        raise RuntimeError("8단계 계획의 JSON 형식을 읽지 못했습니다. 위 원문을 확인하세요.")
    return plan


In [ ]:
# 셀 5: 선택지와 장면 생성. 고정 선택지 유형이나 고정 인물 역할을 사용하지 않습니다.
def stage_of(plan, number):
    return plan["stages"][number - 1]


def available_cast(plan, stage_number):
    return [item for item in plan["cast"] if item["enters_at"] <= stage_number]


def completed_progress(game):
    if not game["state"]["scenes"]:
        return "아직 없음"
    result = []
    for scene in game["state"]["scenes"]:
        stage = stage_of(game["plan"], scene["stage"])
        result.append(f"- {scene['stage']}단계: {stage['progress']}")
    return "\n".join(result)


def plan_context(plan, current_stage):
    stage = stage_of(plan, current_stage)
    next_stage = stage_of(plan, current_stage + 1) if current_stage < 8 else None
    later = [
        {"stage": item["number"], "title": item["title"], "goal": item["goal"], "not_yet": item["not_yet"]}
        for item in plan["stages"][current_stage:]
    ]
    return {
        "title": plan["title"],
        "lesson": plan["lesson"],
        "cast_available_now": available_cast(plan, current_stage),
        "world_rules": plan["world_rules"],
        "threads": plan["threads"],
        "current_stage": stage,
        "next_stage": next_stage,
        "later_outline": later,
    }


def normalize_choices(candidate, raw_text):
    items = candidate.get("choices", []) if isinstance(candidate, dict) else candidate
    if not isinstance(items, list):
        items = []
    choices, seen = [], set()
    for item in items:
        if isinstance(item, str):
            item = {"text": item}
        if not isinstance(item, dict):
            continue
        text = clean_model_text(item.get("text", ""))
        key = re.sub(r"\W+", "", text)
        if len(text) < 4 or len(text) > 40 or key in seen:
            continue
        seen.add(key)
        choices.append({
            "text": text,
            "opening": clean_model_text(item.get("opening", "")),
            "consequence": clean_model_text(item.get("consequence", item.get("bridge", ""))),
        })
        if len(choices) == 3:
            return choices

    # 모델 답변에 JSON이 섞여도 번호 목록/문장 자체는 가능한 한 살려 진행을 막지 않습니다.
    for line in clean_model_text(raw_text).splitlines():
        text = re.sub(r"^\s*(?:[-*]|\d+[.)])\s*", "", line).strip()
        key = re.sub(r"\W+", "", text)
        if 5 <= len(text) <= 40 and key not in seen:
            seen.add(key)
            choices.append({"text": text, "opening": "", "consequence": ""})
        if len(choices) == 3:
            break
    return choices


def repair_choice_format(raw, context):
    prompt = f"""아래 선택지 원문의 행동과 의미는 바꾸지 말고 JSON 형태만 고쳐라.

[원문]\n{raw[:6000]}

[현재 맥락]\n{compact_json(context, 2800)}

{{"choices":[{{"text":"화면에 보일 선택지","opening":"그 선택을 다음 장면 첫 문단에서 실제로 실행한 문장","consequence":"즉시 생긴 결과로 다음 단계에 자연스럽게 이어지는 문장"}}]}}
정확히 세 개를 출력하고 JSON 밖의 말은 쓰지 않는다."""
    repaired = call_llm(prompt, max_tokens=900, temperature=0.05, top_p=0.8)
    return normalize_choices(extract_json_loose(repaired), repaired)


def model_choices(game):
    current = game["state"]["stage"]
    if current >= 8:
        return []
    context = plan_context(game["plan"], current)
    last_scene = game["state"]["scenes"][-1]["text"][-2400:]
    prompt = f"""너는 자연스러운 한국어 인터랙티브 동화의 선택지 작가다.
현재 장면 직후에 아이가 고를 세 가지 행동을 만든다.

[동화 계획]
{compact_json(context)}

[지금까지 확정된 사건]
{completed_progress(game)}

[방금 끝난 장면]
{last_scene}

선택지 규칙:
- 세 선택지는 이번 장면에 나온 인물, 장소, 단서, 물건을 바탕으로 서로 다른 구체적 행동을 제안한다.
- 관찰/대화/도구 사용처럼 코드가 정한 유형을 따르지 말고, 이 장면에 가장 어울리는 서로 다른 방법을 스스로 고른다.
- 각 선택은 다음 단계 목표에 다가가는 방법만 바꾼다. 다음 단계의 해결 결과, 미래 인물의 등장, 최종 결말을 미리 확정하지 않는다.
- 추상적인 "계속한다", "도움을 청한다", "방법을 찾는다"만으로 끝내지 않는다.
- text는 12~40자 사이의 자연스러운 현재형 행동 한 문장이다.
- opening은 text를 실제로 수행한 다음 장면 첫 문단용 문장이다.
- consequence는 opening 바로 다음에 오며, 그 행동의 즉시 결과가 다음 단계로 이어지는 문장이다.
- 선택하지 않은 다른 선택의 행동은 opening/consequence에 섞지 않는다.
- 번호, 이모지, 설명, 마크다운 없이 JSON만 출력한다.

{{"choices":[
  {{"text":"선택 1","opening":"선택 1을 실행한 문장","consequence":"즉시 결과"}},
  {{"text":"선택 2","opening":"선택 2를 실행한 문장","consequence":"즉시 결과"}},
  {{"text":"선택 3","opening":"선택 3을 실행한 문장","consequence":"즉시 결과"}}
]}}"""
    raw = call_llm(prompt, max_tokens=1050, temperature=0.62, top_p=0.92)
    choices = normalize_choices(extract_json_loose(raw), raw)
    if len(choices) < 3:
        print("[정보] 선택지 내용은 유지한 채 JSON 형식만 한 번 복구합니다.")
        choices = repair_choice_format(raw, context)
    if len(choices) < 3:

        # 형식 오류가 있어도 게임을 중단하지 않습니다. 현재 단계의 실제 갈등과
        # 다음 단계 목표를 바탕으로 최소한의 진행 가능한 선택지만 보완합니다.
        print("[정보] 일부 선택지 형식을 읽지 못해 현재 계획 기반 선택지로 보완합니다.")
        current_stage = stage_of(game["plan"], current)
        next_stage = stage_of(game["plan"], current + 1)
        subject = available_cast(game["plan"], current)[0]["name"] if available_cast(game["plan"], current) else "주인공"
        anchors = [
            current_stage.get("conflict") or "주변의 단서",
            current_stage.get("goal") or "다음 길",
            next_stage.get("setting") or "앞으로 갈 곳",
        ]
        verbs = ["자세히 살펴본다", "차분히 확인한다", "조심스럽게 찾아간다"]
        for anchor, verb in zip(anchors, verbs):
            if len(choices) >= 3:
                break
            visible = clean_model_text(f"{subject} {anchor} {verb}.")[:40].rstrip()
            key = re.sub(r"\W+", "", visible)
            if len(visible) >= 4 and key not in {re.sub(r"\W+", "", item["text"]) for item in choices}:
                choices.append({
                    "text": visible,
                    "opening": f"{subject} {anchor} {verb}.",
                    "consequence": f"그 과정에서 다음 단계로 이어질 실마리가 조금 더 또렷해졌다.",
                })

    # Flutter가 서버와 다른 임시 선택지를 보내지 않도록 최종적으로 세 개를 보장합니다.
    if len(choices) < 3:
        current_stage = stage_of(game["plan"], current)
        next_stage = stage_of(game["plan"], current + 1)
        subject = available_cast(game["plan"], current)[0]["name"] if available_cast(game["plan"], current) else "주인공"
        anchors = [
            current_stage.get("conflict") or "주변의 단서",
            current_stage.get("goal") or "앞에 놓인 길",
            next_stage.get("setting") or "다음 장소",
        ]
        endings = ["을 조심히 살펴본다.", "에 남은 흔적을 확인한다.", "의 뜻을 차분히 알아본다.", "을 따라 다음 길을 찾는다."]
        known = {re.sub(r"\W+", "", item["text"]) for item in choices}
        for attempt in range(12):
            if len(choices) >= 3:
                break
            visible = clean_model_text(f"{subject} {anchors[attempt % len(anchors)]}{endings[attempt % len(endings)]}")[:40]
            key = re.sub(r"\W+", "", visible)
            if len(visible) >= 4 and key not in known:
                known.add(key)
                choices.append({
                    "text": visible,
                    "opening": visible,
                    "consequence": "그 행동 덕분에 다음에 해야 할 일이 조금 더 분명해졌다.",
                })
    # Flutter가 서버와 다른 임시 선택지를 보내지 않도록 최종적으로 세 개를 보장합니다.
    if len(choices) < 3:
        current_stage = stage_of(game["plan"], current)
        next_stage = stage_of(game["plan"], current + 1)
        subject = available_cast(game["plan"], current)[0]["name"] if available_cast(game["plan"], current) else "주인공"
        anchors = [
            current_stage.get("conflict") or "주변의 단서",
            current_stage.get("goal") or "앞에 놓인 길",
            next_stage.get("setting") or "다음 장소",
        ]
        endings = ["을 조심히 살펴본다.", "에 남은 흔적을 확인한다.", "의 뜻을 차분히 알아본다.", "을 따라 다음 길을 찾는다."]
        known = {re.sub(r"\W+", "", item["text"]) for item in choices}
        for attempt in range(12):
            if len(choices) >= 3:
                break
            visible = clean_model_text(f"{subject} {anchors[attempt % len(anchors)]}{endings[attempt % len(endings)]}")[:40]
            key = re.sub(r"\W+", "", visible)
            if len(visible) >= 4 and key not in known:
                known.add(key)
                choices.append({
                    "text": visible,
                    "opening": visible,
                    "consequence": "그 행동 덕분에 다음에 해야 할 일이 조금 더 분명해졌다.",
                })
    return choices[:3]


def write_scene(game, stage_number, selected=None):
    plan = game["plan"]
    context = plan_context(plan, stage_number)
    current = stage_of(plan, stage_number)
    previous = game["state"]["scenes"][-1]["text"][-2600:] if game["state"]["scenes"] else "이야기의 시작이다."
    bound_action = "첫 장면이므로 이전 선택은 없다."
    if selected:
        bound_action = f"{selected['opening']}\n{selected['consequence']}"
    stage_end_rule = (
        "이번 단계에서 가장 큰 갈등을 실제 행동과 대화로 해결하고 목표가 이루어졌음을 독자가 분명히 알게 한다."
        if stage_number == 7 else
        "새 위기를 만들지 말고 해결 뒤 달라진 관계와 일상을 보여 주며 마지막 문장에서 이야기를 완전히 끝낸다."
        if stage_number == 8 else
        "이번 단계의 진전까지만 보여 주고 이후 단계의 핵심 해결을 앞당기지 않는다."
    )
    prompt = f"""너는 초등학교 1~3학년을 위한 자연스러운 한국어 창작동화 작가다.

[전체 계획]
{compact_json(context)}

[지금까지 확정된 사건]
{completed_progress(game)}

[직전 장면]
{previous}

[선택으로 확정된 이번 장면의 첫 행동]
{bound_action}

[이번 장면]
- 단계: {stage_number}/8
- 제목: {current['title']}
- 장소: {current['setting']}
- 이번 단계 목표: {current['goal']}
- 독자의 질문: {current['question']}
- 장애물 또는 갈등: {current['conflict']}
- 장면 뒤 확정 사실: {current['progress']}
- 아직 일어나면 안 되는 일: {current['not_yet']}

작성 규칙:
- 선택으로 확정된 첫 행동이 있다면 그것을 첫 문단 첫 두 문장 안에서 실제로 실행한다. 행동을 취소하거나 다른 행동으로 바꾸지 않는다.
- 그 선택이 이번 단계의 목표와 갈등으로 이어지는 원인이 되게 쓴다.
- 계획의 cast에 정해진 인물·중요 존재만 실제 이름을 갖고 등장하거나 행동하게 한다. 이름 없는 군중·동물·마을 사람은 분위기 묘사로만 짧게 쓸 수 있으며, 새 이름·새 핵심 인물·새 핵심 물건을 만들지 않는다.
- 모델이 계획한 인물의 성격, 바람, 등장 시점을 끝까지 지킨다.
- 세계 규칙과 이미 나온 단서를 잊지 말고, 계획표를 독자에게 설명하지 않는다.
- 행동, 감각 묘사, 짧은 대화로 장면을 보여 준다. 같은 격려·감탄·문장을 반복하지 않는다.
- 잔혹한 폭력, 죽음, 과도한 공포, 게임 용어, 이모지, 제목, 단계 번호, 해시태그, 코드, 다음 장 예고를 쓰지 않는다.
- 3~4문단, 12~18문장으로 충분히 길고 매끄럽게 쓴다.
- {stage_end_rule}

동화 본문만 출력하라."""
    raw = call_llm(prompt, max_tokens=1250, temperature=0.66, top_p=0.91)
    text = clean_story(raw, headings=[plan["title"], current["title"]])
    if selected:
        # 생성 모델이 선택 문장을 생략해도 선택의 원인을 잃지 않게, 모델이 만든 연결 문장만 본문 앞에 둡니다.
        prefix = "\n\n".join(part for part in (selected["opening"], selected["consequence"]) if part)
        if prefix and selected["opening"] not in text:
            text = clean_story(prefix + "\n\n" + text)
    return text


In [ ]:
# 셀 6: 상태 머신. 선택 행동과 그 선택의 Qwen 감정 점수를 함께 다음 장면으로 넘깁니다.
import copy
import random
import secrets


def new_game(plan, genre, age):
    return {
        "plan": copy.deepcopy(plan),
        "settings": {"genre": genre, "age": age},
        "state": {
            "stage": 1,
            "ended": False,
            "scenes": [],
            "choices": [],
            "history": [],
        },
    }


def start_game(plan, genre="판타지", age="초등학교 1~3학년"):
    game = new_game(plan, genre, age)
    text = write_scene(game, 1)
    game["state"]["scenes"].append({"stage": 1, "title": stage_of(plan, 1)["title"], "text": text})
    game["state"]["choices"] = model_choices(game)
    return game


def all_story_text(game):
    return "\n\n".join(scene["text"] for scene in game["state"]["scenes"])


def choose_and_advance(game, choice_index):
    if choice_index not in (0, 1, 2):
        raise ValueError("선택지 번호는 0, 1, 2 중 하나여야 합니다.")
    if game["state"]["ended"]:
        raise ValueError("이미 8단계 결말까지 완료된 동화입니다.")
    updated = copy.deepcopy(game)
    selected = copy.deepcopy(updated["state"]["choices"][choice_index])
    previous_scene = updated["state"]["scenes"][-1]["text"]
    selected.setdefault("emotion", analyze_choice_emotion(previous_scene, selected["text"]))
    next_stage = updated["state"]["stage"] + 1
    updated["state"]["stage"] = next_stage
    text = write_scene(updated, next_stage, selected)
    updated["state"]["scenes"].append({
        "stage": next_stage,
        "title": stage_of(updated["plan"], next_stage)["title"],
        "text": text,
        "emotion": analyze_story_emotion(text),
    })
    updated["state"]["history"].append({
        "from_stage": next_stage - 1,
        "to_stage": next_stage,
        "choice": selected,
    })
    if next_stage == 8:
        updated["state"]["ended"] = True
        updated["state"]["choices"] = []
    else:
        updated["state"]["choices"] = model_choices(updated)
    return updated


In [ ]:
# 셀 7: Flutter 연동 FastAPI 서버. 셀 1~6 실행 뒤 이 셀을 실행하면 됩니다.
# 같은 메모리의 Qwen 계획 엔진을 사용하므로 별도의 모델 로딩이 필요하지 않습니다.

import base64
import os
import re
import secrets
import subprocess
import threading
import time
from io import BytesIO
from tempfile import NamedTemporaryFile, TemporaryDirectory

import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, Response


ACTIVE_GAMES = {}
STT_MODEL = None
STT_MODEL_LOCK = threading.Lock()


def ensure_stt_model():
    global STT_MODEL
    if STT_MODEL is not None:
        return STT_MODEL
    with STT_MODEL_LOCK:
        if STT_MODEL is None:
            from faster_whisper import WhisperModel
            STT_MODEL = WhisperModel(
                STT_MODEL_NAME,
                device=STT_DEVICE,
                compute_type=STT_COMPUTE_TYPE,
                download_root=STT_CACHE_ROOT,
            )
    return STT_MODEL


def encode_runtime_state(game):
    raw = json.dumps(
        {"version": 1, "game": game},
        ensure_ascii=False,
        separators=(",", ":"),
    ).encode("utf-8")
    return base64.urlsafe_b64encode(raw).decode("ascii")


def decode_runtime_state(token):
    if not isinstance(token, str) or not token.strip():
        return None
    try:
        raw = base64.urlsafe_b64decode(token.encode("ascii"))
        if len(raw) > 250_000:
            return None
        payload = json.loads(raw.decode("utf-8"))
        game = payload.get("game") if isinstance(payload, dict) else None
        if not isinstance(game, dict):
            return None
        if not isinstance(game.get("plan"), dict) or not isinstance(game.get("state"), dict):
            return None
        if not isinstance(game["state"].get("scenes"), list):
            return None
        return game
    except Exception:
        return None


api_app = FastAPI(title="Fairytale test_v1 API")
api_app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)


def api_error(status, message):
    raise HTTPException(status_code=status, detail=message)


def choice_emotion_payload(game):
    previous_scene = game["state"]["scenes"][-1]["text"]
    payload = []
    for choice in game["state"]["choices"]:
        choice.setdefault("emotion", analyze_choice_emotion(previous_scene, choice["text"]))
        payload.append(choice["emotion"])
    return payload


def story_characters_payload(game):
    emojis = ["🌟", "🦊", "🦉", "🐉", "🌿", "✨", "🗺️"]
    profiles = []
    for index, item in enumerate(game["plan"]["cast"][:5]):
        profiles.append({
            "name": item["name"],
            "role": "이야기 속 중요한 인물",
            "personality": item["description"],
            "greeting": f"안녕, 나는 {item['name']}이야. 우리 이야기에서 궁금한 것이 있니?",
            "avatar_emoji": emojis[index % len(emojis)],
        })
    return profiles


def game_for(story_id, runtime_state=None):
    game = ACTIVE_GAMES.get(story_id)
    if game is not None:
        return game
    game = decode_runtime_state(runtime_state)
    if game is None:
        api_error(404, "이어 읽기 정보가 없어요. 새 동화를 먼저 시작해 주세요.")
    ACTIVE_GAMES[story_id] = game
    return game


def response_for_start(story_id, game):
    first = game["state"]["scenes"][0]
    first.setdefault("emotion", analyze_story_emotion(first["text"]))
    return {
        "story_id": story_id,
        "story_text": first["text"],
        "choices": [item["text"] for item in game["state"]["choices"]],
        "choice_emotions": choice_emotion_payload(game),
        "story_emotion": first["emotion"],
        "vocab": [],
        "chapter": 1,
        "image_b64": None,
        "characters": {item["name"]: item["description"] for item in game["plan"]["cast"]},
        "runtime_state": encode_runtime_state(game),
        "completed": False,
    }


@api_app.get("/")
def api_root():
    return {"status": "ok", "service": "fairytale test_v1"}


@api_app.get("/health")
def api_health():
    return {
        "status": "ok",
        "engine": "Unsloth Qwen3.8-27B GGUF",
        "emotion_model": "Qwen contextual analysis",
        "active_stories": len(ACTIVE_GAMES),
        "tts": "XTTS-v2 voice cloning",
        "stt": f"faster-whisper {STT_MODEL_NAME} ({'ready' if STT_MODEL is not None else 'lazy'})",
    }


@api_app.post("/story/start")
async def api_start(request: Request):
    body = await request.json()
    prompt = clean_model_text(body.get("prompt", "")) or "따뜻한 모험 동화"
    genre = clean_model_text(body.get("genre", "")) or "판타지"
    age = clean_model_text(body.get("age", "")) or "초등학교 1~3학년"
    plan = make_story_plan(prompt, age)
    game = start_game(plan, genre, age)
    story_id = "testv1_" + secrets.token_urlsafe(12)
    ACTIVE_GAMES[story_id] = game
    return response_for_start(story_id, game)


@api_app.post("/story/continue")
async def api_continue(request: Request):
    body = await request.json()
    story_id = str(body.get("story_id", ""))
    game = game_for(story_id, body.get("runtime_state"))
    if game["state"]["ended"]:
        api_error(400, "이미 완결된 동화입니다.")
    requested_choice = clean_model_text(body.get("choice", ""))
    current_choices = game["state"]["choices"]
    choice_index = next((index for index, item in enumerate(current_choices) if item["text"] == requested_choice), None)
    if choice_index is None:
        api_error(400, "현재 장면의 선택지를 골라 주세요.")
    updated = choose_and_advance(game, choice_index)
    ACTIVE_GAMES[story_id] = updated
    scene = updated["state"]["scenes"][-1]
    selected = updated["state"]["history"][-1]["choice"]
    next_emotions = choice_emotion_payload(updated) if not updated["state"]["ended"] else []
    return {
        "story_id": story_id,
        "new_text": scene["text"],
        "choices": [item["text"] for item in updated["state"]["choices"]],
        "choice_emotions": next_emotions,
        "selected_choice_emotion": selected.get("emotion"),
        "story_emotion": scene.get("emotion") or analyze_story_emotion(scene["text"]),
        "vocab": [],
        "chapter": updated["state"]["stage"],
        "image_b64": None,
        "runtime_state": encode_runtime_state(updated),
        "completed": updated["state"]["ended"],
    }


def _psych_story_context(game):
    if game is None:
        return "동화 원문은 현재 서버에 남아 있지 않습니다. 선택 기록만 근거로 해설합니다."
    scenes = []
    for scene in game["state"]["scenes"]:
        scenes.append({
            "stage": scene.get("stage"),
            "title": scene.get("title"),
            "scene": clean_story(scene.get("text", ""))[:720],
        })
    return compact_json({
        "lesson": game["plan"].get("lesson", ""),
        "stages": scenes,
        "selected_paths": [
            item.get("choice", {}).get("text", "")
            for item in game["state"].get("history", [])
        ],
    }, limit=6400)


def api_psych_prompt(title, choices, emotions, story_context):
    evidence = []
    for index, choice in enumerate(choices):
        emotion = emotions[index] if index < len(emotions) and isinstance(emotions[index], dict) else {}
        evidence.append({
            "step": index + 1,
            "choice": clean_model_text(choice),
            "primary_emotion": emotion.get("primary_emotion", ""),
            "primary_score": emotion.get("primary_score", 0),
            "top_emotions": emotion.get("top_emotions", [])[:3],
        })
    return f'''너는 아동 발달과 놀이 관찰의 원칙을 아는 어린이 동화 앱 해설가다. 이것은 동화 속 선택을 활용한 대화용 해설이지, 실제 아동의 성격·지능·정신건강을 판단하거나 진단하는 검사가 아니다.

[동화 제목]
{title}

[동화 장면과 선택 맥락]
{story_context}

[사용자가 실제로 고른 선택과 서사적 감정 점수]
{json.dumps(evidence, ensure_ascii=False)}

작성 원칙:
- 일반적인 칭찬, '따뜻한 여정', '세계와 소통'처럼 근거 없는 문구를 쓰지 않는다.
- 반드시 실제 선택 문장을 2개 이상 짧게 인용하고, 해당 장면의 감정 점수를 함께 연결한다.
- 관찰 사실과 가능한 의미를 구분한다. '이 선택은 ~로 해석될 여지가 있다', '동화 안에서는'처럼 제한적으로 쓴다.
- 한 번의 동화 선택으로 현실의 아이를 단정하지 않는다. 부정적인 선택도 문제로 낙인찍지 않는다.
- description은 아래 세 소제목을 포함한 7~9문장 한국어로 쓴다.
  [동화 속 관찰] 선택과 장면의 구체적 사실 2개 이상.
  [가능한 의미] 놀이·서사 안에서 살펴볼 수 있는 경향 2개.
  [함께 이야기해 보기] 어른이 아이와 나눌 수 있는 열린 질문 1개.
- traits는 성격 점수가 아니라 이번 동화의 선택 경향 지표다. 35~75 사이 정수로 과장 없이 작성한다.
- choice_insights에는 각 선택마다 '관찰: ... / 가능한 의미: ...' 형식의 한 문장을 순서대로 작성한다.
- caregiver_prompts에는 정답을 유도하지 않는 열린 질문 3개를 넣는다.
- analysis_caution에는 '동화 속 선택을 바탕으로 한 대화용 해설이며 심리검사가 아닙니다.'와 같은 문장을 넣는다.
- JSON 외의 말이나 코드블록은 쓰지 않는다.
{{"type":"동화 속 선택 관찰","description":"[동화 속 관찰] ...\n\n[가능한 의미] ...\n\n[함께 이야기해 보기] ...","traits":{{"모험적":50,"친절함":50,"용감함":50,"창의적":50,"협동심":50}},"dominant_emotions":["감정1","감정2","감정3"],"choice_insights":["관찰: ... / 가능한 의미: ..."],"caregiver_prompts":["열린 질문1","열린 질문2","열린 질문3"],"analysis_caution":"동화 속 선택을 바탕으로 한 대화용 해설이며 심리검사가 아닙니다."}}'''


def normalize_api_psych(raw, choices, emotions):
    data = extract_json_loose(raw)
    if not isinstance(data, dict):
        data = {}
    labels = []
    for emotion in emotions:
        if not isinstance(emotion, dict):
            continue
        for item in emotion.get("top_emotions", [])[:3]:
            if isinstance(item, dict) and item.get("label") and item["label"] not in labels:
                labels.append(item["label"])
    traits = data.get("traits") if isinstance(data.get("traits"), dict) else {}
    normalized_traits = {}
    for name in ["모험적", "친절함", "용감함", "창의적", "협동심"]:
        try:
            normalized_traits[name] = max(35, min(75, int(float(traits.get(name, 50)))))
        except Exception:
            normalized_traits[name] = 50
    insights = data.get("choice_insights") if isinstance(data.get("choice_insights"), list) else []
    insights = [clean_model_text(item) for item in insights if clean_model_text(item)]
    if not insights:
        insights = [
            f"관찰: '{choice}'을 골랐어요. / 가능한 의미: 이 장면에서 어떤 방법을 먼저 살피고 싶었는지 함께 물어볼 수 있어요."
            for choice in choices
        ]
    description = clean_model_text(data.get("description", "")) or clean_model_text(raw)
    if len(description) < 80:
        first = clean_model_text(choices[0]) if choices else "다음 길을 고른 장면"
        second = clean_model_text(choices[1]) if len(choices) > 1 else first
        description = (
            f"[동화 속 관찰] 주인공은 '{first}'과 '{second}'을 선택하며 장면을 이어 갔어요. "
            "각 선택에는 그때의 이야기 흐름과 감정 점수가 함께 기록되어 있어요.\n\n"
            "[가능한 의미] 이 기록은 이번 동화에서 어떤 해결 방법에 마음이 갔는지 대화로 살펴볼 단서가 될 수 있어요. "
            "한 편의 동화만으로 현실의 성격을 판단할 수는 없어요.\n\n"
            "[함께 이야기해 보기] '그 선택을 고른다면 다음에는 어떤 일이 일어날 것 같았어?'라고 편하게 물어보세요."
        )
    prompts = data.get("caregiver_prompts") if isinstance(data.get("caregiver_prompts"), list) else []
    prompts = [clean_model_text(item) for item in prompts if clean_model_text(item)][:3]
    if len(prompts) < 3:
        prompts = [
            "이야기에서 가장 오래 생각한 선택은 무엇이었어?",
            "그 선택을 고르면 다음에 어떤 일이 생길 것 같았어?",
            "다시 읽는다면 다른 방법도 떠오르니?",
        ]
    return {
        "type": clean_model_text(data.get("type", "")) or "동화 속 선택 관찰",
        "description": description,
        "traits": normalized_traits,
        "dominant_emotions": data.get("dominant_emotions", labels) or labels,
        "choice_insights": insights[:max(1, len(choices))],
        "caregiver_prompts": prompts,
        "analysis_caution": clean_model_text(data.get("analysis_caution", "")) or "동화 속 선택을 바탕으로 한 대화용 해설이며 심리검사가 아닙니다.",
    }


@api_app.post("/story/psych")
async def api_psych(request: Request):
    body = await request.json()
    story_id = str(body.get("story_id", ""))
    game = ACTIVE_GAMES.get(story_id)
    choices = body.get("choices_made") if isinstance(body.get("choices_made"), list) else []
    emotions = body.get("choice_emotions") if isinstance(body.get("choice_emotions"), list) else []
    raw = call_llm(
        api_psych_prompt(
            clean_model_text(body.get("story_title", "")) or "완성된 동화",
            choices,
            emotions,
            _psych_story_context(game),
        ),
        1250,
        0.25,
        0.85,
    )
    return normalize_api_psych(raw, choices, emotions)


@api_app.post("/story/characters")
async def api_characters(request: Request):
    body = await request.json()
    game = game_for(str(body.get("story_id", "")))
    return {"characters": story_characters_payload(game)}


def character_chat_context(body):
    # Live games use the full plan; saved and mock stories use Flutter context.
    story_id = str(body.get("story_id", ""))
    game = ACTIVE_GAMES.get(story_id)
    character = body.get("character") if isinstance(body.get("character"), dict) else {}
    character_name = clean_model_text(character.get("name", ""))
    if not character_name:
        api_error(400, "대화할 캐릭터 이름이 필요합니다.")

    if game is not None:
        profile = next((item for item in game["plan"]["cast"] if item["name"] == character_name), None)
        if profile is None:
            api_error(400, "이 동화 계획에 없는 캐릭터입니다.")
        return profile, game["plan"], all_story_text(game)[-11000:], "현재 서버의 동화 계획"

    # The Flutter client includes these values so a server restart must not
    # turn an already-read story into a generic local-only chat.
    story_text = clean_model_text(body.get("story_text", ""))[-12000:]
    if not story_text:
        api_error(400, "서버에 동화 기록이 없고 전달된 동화 본문도 비어 있습니다.")
    profile = {
        "name": character_name,
        "role": clean_model_text(character.get("role", "")) or "이야기 속 친구",
        "description": clean_model_text(character.get("personality", "")) or "동화의 사건을 함께 겪은 다정한 친구",
        "greeting": clean_model_text(character.get("greeting", "")),
    }
    recovered_plan = {
        "title": clean_model_text(body.get("story_title", "")) or "저장된 동화",
        "cast": [profile],
        "note": "서버 재시작 뒤 Flutter가 전달한 완성 동화 본문으로 복원한 대화 맥락입니다.",
    }
    return profile, recovered_plan, story_text, "Flutter가 전달한 저장 동화 본문"


@api_app.post("/story/character-chat")
async def api_character_chat(request: Request):
    body = await request.json()
    profile, plan_context, story_text, context_source = character_chat_context(body)
    character_name = profile["name"]
    messages = body.get("messages") if isinstance(body.get("messages"), list) else []
    user_name = clean_model_text(body.get("user_name", "")) or "동화 친구"
    user_message = clean_model_text(body.get("user_message", ""))
    if not user_message:
        api_error(400, "보낼 메시지를 입력해 주세요.")
    history_text = "\n".join(
        f"{'아이' if item.get('role') == 'user' else character_name}: {clean_model_text(item.get('content', ''))[:400]}"
        for item in messages[-12:] if isinstance(item, dict)
    ) or "아직 대화 없음"
    prompt = f'''너는 어린이 동화의 캐릭터 역할극을 한다. 반드시 {character_name}으로만 1인칭으로 답한다.

[동화 계획]
{compact_json(plan_context)}

[동화 전문 - {context_source}]
{story_text}

[현재 캐릭터]
{json.dumps(profile, ensure_ascii=False)}

[이전 대화]
{history_text}

[아이 이름] {user_name}
[아이 메시지] {user_message}

규칙:
- 캐릭터는 위 동화에서 실제로 겪은 사건, 관계, 결말을 기억하고 그것과 모순되지 않게 답한다.
- 동화 본문에 없는 일을 사실처럼 꾸며내지 않는다. 모르면 '내가 이야기에서 보지 못한 일이야'라고 솔직히 말한다.
- AI, 모델, 프롬프트, 서버라는 말은 하지 않는다.
- 캐릭터 성격과 동화의 결말을 뒤집지 않는다.
- 어린이가 이해할 쉬운 한국어 2~4문장으로 답한다.
- suggested_replies에는 서로 다른 짧은 다음 질문 세 개를 넣는다.
- JSON 외의 말이나 코드블록은 쓰지 않는다.
{{"reply":"캐릭터 답장","suggested_replies":["질문1","질문2","질문3"]}}'''
    raw = call_llm(prompt, 520, 0.65, 0.92)
    data = extract_json_loose(raw)
    if not isinstance(data, dict):
        data = {}
    reply = clean_model_text(data.get("reply", "")) or clean_model_text(raw)
    suggestions = data.get("suggested_replies") if isinstance(data.get("suggested_replies"), list) else []
    return {
        "reply": reply[:1200],
        "suggested_replies": [clean_model_text(item)[:80] for item in suggestions if clean_model_text(item)][:3],
    }


@api_app.post("/story/image")
async def api_image(request: Request):
    body = await request.json()
    try:
        image = generate_illustration(clean_model_text(body.get("story_text", "")), clean_model_text(body.get("genre", "")) or "판타지")
        buffer = BytesIO()
        image.save(buffer, format="PNG")
        return {"image_b64": base64.b64encode(buffer.getvalue()).decode("ascii")}
    except Exception as error:
        api_error(503, f"삽화 생성 실패: {error}")


@api_app.post("/api/stt/warm-up")
def api_stt_warm_up():
    try:
        ensure_stt_model()
        return {"status": "ready", "engine": f"faster-whisper {STT_MODEL_NAME}", "device": STT_DEVICE}
    except Exception as error:
        api_error(503, f"음성 인식 준비 실패: {error}")


@api_app.post("/api/stt")
async def api_stt(request: Request):
    form = await request.form()
    uploaded = form.get("audio") or form.get("audio_file")
    if uploaded is None or not hasattr(uploaded, "read"):
        api_error(400, "audio WAV 파일을 함께 보내 주세요.")
    audio = await uploaded.read()
    if len(audio) < 800:
        api_error(400, "녹음이 너무 짧습니다.")
    language = clean_model_text(form.get("language", "ko")) or "ko"
    suffix = Path(getattr(uploaded, "filename", "voice.wav")).suffix or ".wav"
    temporary_path = None
    try:
        with NamedTemporaryFile(suffix=suffix, delete=False) as temporary_file:
            temporary_file.write(audio)
            temporary_path = temporary_file.name
        model = ensure_stt_model()
        segments, info = model.transcribe(
            temporary_path,
            language=language,
            task="transcribe",
            beam_size=5,
            vad_filter=True,
            condition_on_previous_text=False,
            initial_prompt="어린이 동화 속 캐릭터에게 하는 한국어 질문입니다.",
        )
        text = clean_model_text(" ".join(segment.text for segment in segments))
        if not text:
            api_error(422, "음성을 이해하지 못했어요. 더 또렷하게 말해 주세요.")
        return {
            "text": text,
            "language": language,
            "duration": round(float(getattr(info, "duration", 0.0)), 2),
            "engine": f"faster-whisper {STT_MODEL_NAME}",
        }
    except HTTPException:
        raise
    except Exception as error:
        api_error(503, f"음성 인식 실패: {error}")
    finally:
        if temporary_path:
            try:
                os.unlink(temporary_path)
            except FileNotFoundError:
                pass


@api_app.post("/api/tts/warm-up")
def api_tts_warm_up():
    try:
        ensure_xtts_sidecar()
        return {"status": "ready", "engine": "XTTS-v2", "device": XTTS_DEVICE}
    except Exception as error:
        api_error(503, f"XTTS 준비 실패: {error}")


@api_app.post("/api/tts")
async def api_tts(request: Request):
    content_type = request.headers.get("content-type", "")
    sample = None
    if content_type.startswith("multipart/"):
        form = await request.form()
        text = clean_model_text(form.get("text", ""))
        uploaded = form.get("speaker_wav")
        if uploaded is not None and hasattr(uploaded, "read"):
            sample = await uploaded.read()
    else:
        body = await request.json()
        text = clean_model_text(body.get("text", ""))
        encoded = body.get("speaker_wav_b64", "")
        if encoded:
            try:
                sample = base64.b64decode(encoded, validate=True)
            except Exception:
                api_error(400, "speaker_wav_b64가 올바른 WAV 데이터가 아닙니다.")
    if not text:
        api_error(400, "읽을 동화 텍스트가 비어 있습니다.")
    if not sample:
        api_error(400, "XTTS-v2는 내 목소리 녹음 WAV가 필요합니다.")
    if sample[:4] != b"RIFF" or sample[8:12] != b"WAVE":
        api_error(400, "PCM WAV 형식의 내 목소리 녹음 파일이 필요합니다.")
    try:
        with TemporaryDirectory() as directory:
            speaker_path = Path(directory) / "speaker.wav"
            speaker_path.write_bytes(sample)
            audio = synthesize_with_my_voice(text, speaker_path)
        return Response(content=audio, media_type="audio/wav")
    except Exception as error:
        api_error(503, f"XTTS 음성 합성 실패: {error}")


if "api_server_thread" not in globals() or not api_server_thread.is_alive():
    nest_asyncio.apply()
    api_config = uvicorn.Config(api_app, host="127.0.0.1", port=8000, log_level="warning")
    api_server = uvicorn.Server(api_config)
    api_server_thread = threading.Thread(target=api_server.run, daemon=True)
    api_server_thread.start()
    time.sleep(2)
    print("FastAPI 서버를 127.0.0.1:8000에서 시작했습니다.")
else:
    print("FastAPI 서버가 이미 실행 중입니다.")


TUNNEL_LOG_PATH = Path("/tmp/test_v1_cloudflared.log")
if "api_tunnel_process" not in globals() or api_tunnel_process.poll() is not None:
    if TUNNEL_LOG_PATH.exists():
        TUNNEL_LOG_PATH.unlink()
    api_tunnel_process = subprocess.Popen(
        [
            "cloudflared", "tunnel", "--url", "http://127.0.0.1:8000",
            "--no-autoupdate", "--logfile", str(TUNNEL_LOG_PATH), "--loglevel", "info",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    public_url = ""
    for _ in range(90):
        time.sleep(1)
        if TUNNEL_LOG_PATH.exists():
            matched = re.search(
                r"https://[-a-z0-9]+\.trycloudflare\.com",
                TUNNEL_LOG_PATH.read_text(errors="replace"),
            )
            if matched:
                public_url = matched.group(0)
                break
    if not public_url:
        details = TUNNEL_LOG_PATH.read_text(errors="replace")[-3000:] if TUNNEL_LOG_PATH.exists() else ""
        raise RuntimeError("Cloudflare 터널 URL을 받지 못했습니다.\n" + details)
    API_PUBLIC_URL = public_url
elif "API_PUBLIC_URL" not in globals():
    raise RuntimeError("이미 실행 중인 터널의 URL을 알 수 없습니다. 런타임을 다시 시작한 뒤 셀 7을 실행하세요.")

print("=" * 68)
print("Flutter .env에 아래 두 줄을 넣으세요.")
print(f"AI_API_BASE_URL={API_PUBLIC_URL}")
print(f"TTS_API_BASE_URL={API_PUBLIC_URL}")
print("=" * 68)


In [ ]:
# 셀 8: 요청 하나로 계획을 만들고, 7번의 무작위 선택과 감정 기록을 반영해 8단계 결말까지 생성합니다.
USER_REQUEST = "용에게 납치된 공주님을 구하러 가는 어린 용사의 따뜻한 모험 동화"
READER_AGE = "초등학교 1~3학년"
STORY_GENRE = "판타지"
RANDOM_CHOICE_SEED = secrets.randbits(64)
rng = random.Random(RANDOM_CHOICE_SEED)

STORY_PLAN = make_story_plan(USER_REQUEST, READER_AGE)
print("[제목]", STORY_PLAN["title"])
print("[교훈]", STORY_PLAN["lesson"])
print("[미리 정해진 등장인물/중요 존재]", json.dumps(STORY_PLAN["cast"], ensure_ascii=False, indent=2))
print("[8단계 계획]")
for stage in STORY_PLAN["stages"]:
    print(f"{stage['number']}. {stage['title']} | {stage['goal']}")

game = start_game(STORY_PLAN, STORY_GENRE, READER_AGE)
print("\n[무작위 선택 시드]", RANDOM_CHOICE_SEED)

while True:
    scene = game["state"]["scenes"][-1]
    print(f"\n===== {scene['stage']}단계: {scene['title']} =====\n")
    print(scene["text"])
    if game["state"]["ended"]:
        break
    print("\n[선택지]")
    for index, choice in enumerate(game["state"]["choices"], 1):
        print(f"{index}. {choice['text']}")
    selected_index = rng.randrange(3)
    selected = game["state"]["choices"][selected_index]
    print(f"\n[무작위 선택] {selected_index + 1}. {selected['text']}")
    game = choose_and_advance(game, selected_index)
    selected_emotion = game["state"]["history"][-1]["choice"]["emotion"]
    print("[선택 감정]", selected_emotion["primary_emotion"], selected_emotion["primary_score"])

assert len(game["state"]["scenes"]) == 8
assert len(game["state"]["history"]) == 7
print("\n[완료] 7번의 선택과 선택별 감정 점수를 반영해 8단계 결말까지 생성했습니다.")


In [ ]:
# 셀 9: 선택사항 - 각 장면의 삽화를 만듭니다.
# L4에서는 DreamShaper가 CPU로 동작하므로 오래 걸립니다. 필요한 장면만 True로 바꾸세요.
GENERATE_STAGE_ILLUSTRATIONS = False
ILLUSTRATION_STAGES = [1, 4, 7, 8]

if GENERATE_STAGE_ILLUSTRATIONS:
    from IPython.display import display
    illustrations = {}
    for scene in game["state"]["scenes"]:
        if scene["stage"] not in ILLUSTRATION_STAGES:
            continue
        print(f"삽화 생성: {scene['stage']}단계 {scene['title']}")
        image = generate_illustration(scene["text"], STORY_GENRE)
        illustrations[scene["stage"]] = image
        display(image)
else:
    print("삽화 생성은 건너뛰었습니다. GENERATE_STAGE_ILLUSTRATIONS=True로 바꾸면 DreamShaper 8을 사용합니다.")


In [ ]:
# 셀 10: 선택 기록 + Qwen이 생성한 감정 점수를 근거로 엔딩 분석 문장을 생성합니다.
def p_psych_report(game):
    records = []
    for item in game["state"]["history"]:
        emotion = item["choice"].get("emotion", {})
        records.append({
            "step": item["from_stage"],
            "choice": item["choice"]["text"],
            "primary_emotion": emotion.get("primary_emotion", ""),
            "primary_score": emotion.get("primary_score", 0),
            "top_emotions": emotion.get("top_emotions", [])[:3],
        })
    return f"""너는 아동 발달과 놀이 관찰의 원칙을 아는 동화 해설가다. 이 결과는 동화 속 선택을 바탕으로 한 대화용 해설이며 심리검사가 아니다.

[동화 제목] {game['plan']['title']}
[선택과 감정 근거]
{json.dumps(records, ensure_ascii=False, indent=2)}

규칙:
- 실제 선택 문장 두 개 이상과 그 감정 점수를 반드시 언급한다.
- 관찰 사실과 가능한 의미를 구분하고, 현실의 아이를 단정하거나 진단하지 않는다.
- description은 [동화 속 관찰], [가능한 의미], [함께 이야기해 보기] 소제목을 포함한 7~9문장이다.
- traits는 이번 동화의 선택 경향 지표이며 35~75 사이 정수로 쓴다.
- choice_insights는 '관찰: ... / 가능한 의미: ...' 형식으로 쓴다.
- caregiver_prompts에는 열린 질문 세 개를 넣는다.
- analysis_caution에는 심리검사가 아니라는 안내를 넣는다.
- JSON 외의 말이나 코드블록은 쓰지 않는다.
{{"type":"동화 속 선택 관찰","description":"[동화 속 관찰] ...\n\n[가능한 의미] ...\n\n[함께 이야기해 보기] ...","traits":{{"모험적":50,"친절함":50,"용감함":50,"창의적":50,"협동심":50}},"dominant_emotions":["감정1","감정2","감정3"],"choice_insights":["관찰: ... / 가능한 의미: ..."],"caregiver_prompts":["질문1","질문2","질문3"],"analysis_caution":"동화 속 선택을 바탕으로 한 대화용 해설이며 심리검사가 아닙니다."}}"""


def normalize_psych_report(raw, game):
    data = extract_json_loose(raw)
    if not isinstance(data, dict):
        data = {}
    selected = [item["choice"] for item in game["state"]["history"]]
    labels = []
    for choice in selected:
        for emotion in choice.get("emotion", {}).get("top_emotions", [])[:3]:
            label = str(emotion.get("label", "")).strip()
            if label and label not in labels:
                labels.append(label)
    labels = labels[:3]
    description = clean_model_text(data.get("description", ""))
    if len(description) < 30:
        description = clean_model_text(raw)
    if len(description) < 30:
        description = "선택과 감정 점수에 대한 해설을 만들지 못했습니다. 분석 결과를 다시 생성해 주세요."
    defaults = {"모험적": 55, "친절함": 55, "용감함": 55, "창의적": 55, "협동심": 55}
    traits = data.get("traits", {}) if isinstance(data.get("traits"), dict) else {}
    normalized_traits = {}
    for name, default in defaults.items():
        try:
            normalized_traits[name] = max(0, min(100, int(float(traits.get(name, default)))))
        except Exception:
            normalized_traits[name] = default
    insights = data.get("choice_insights", [])
    insights = [clean_model_text(item) for item in insights if clean_model_text(item)] if isinstance(insights, list) else []
    return {
        "type": clean_model_text(data.get("type", "")) or "이번 이야기의 선택 여정",
        "description": description,
        "traits": normalized_traits,
        "dominant_emotions": data.get("dominant_emotions", labels) or labels,
        "choice_insights": insights,
        "evidence": [
            {"choice": choice["text"], "emotion": choice.get("emotion", {}).get("top_emotions", [])[:3]}
            for choice in selected
        ],
    }


PSYCH_REPORT = normalize_psych_report(call_llm(p_psych_report(game), max_tokens=850, temperature=0.45, top_p=0.9), game)
print("[분석 유형]", PSYCH_REPORT["type"])
print("[감정]", ", ".join(PSYCH_REPORT["dominant_emotions"]))
print("\n" + PSYCH_REPORT["description"])
print("\n[선택별 해설]")
for insight in PSYCH_REPORT["choice_insights"]:
    print("-", insight)


In [ ]:
# 셀 11: 계획에 미리 정해진 등장인물과 대화합니다.
# 캐릭터는 전체 계획과 완성된 8개 장면을 함께 받아 동화 속 사건을 기억합니다.
def character_profiles(game):
    return [
        {
            "name": item["name"],
            "description": item["description"],
            "desire": item["desire"],
            "enters_at": item["enters_at"],
        }
        for item in game["plan"]["cast"]
    ]


def character_chat(game, character_name, user_message, history=None, user_name="동화 친구"):
    profile = next((item for item in character_profiles(game) if item["name"] == character_name), None)
    if profile is None:
        raise ValueError("계획에 없는 캐릭터입니다. character_profiles(game) 목록에서 골라주세요.")
    history = history or []
    history_text = "\n".join(
        f"{'아이' if item.get('role') == 'user' else character_name}: {str(item.get('content', ''))[:400]}"
        for item in history[-12:] if isinstance(item, dict)
    ) or "아직 대화 없음"
    prompt = f"""너는 완성된 어린이 동화의 캐릭터 역할극을 한다. 아래 프로필의 {character_name}으로만 1인칭 대답을 한다.

[동화 계획]
{compact_json(game['plan'])}

[완성된 동화]
{all_story_text(game)[-11000:]}

[현재 캐릭터]
{json.dumps(profile, ensure_ascii=False)}

[이전 대화]
{history_text}

[아이 이름] {user_name}
[아이 메시지] {user_message}

규칙:
- 캐릭터는 동화에서 실제로 겪은 사건과 관계를 기억한다.
- AI, 모델, 프롬프트, 설정이라는 말은 하지 않는다.
- 캐릭터 성격과 결말을 뒤집지 않는다.
- 어린이가 이해할 쉬운 한국어 2~4문장으로 답한다.
- suggested_replies에는 서로 다른 짧은 다음 질문 세 개를 넣는다.
- JSON 외의 말은 쓰지 않는다.
{{"reply":"캐릭터 답장","suggested_replies":["질문1","질문2","질문3"]}}"""
    raw = call_llm(prompt, max_tokens=520, temperature=0.65, top_p=0.92)
    data = extract_json_loose(raw)
    if not isinstance(data, dict):
        data = {}
    reply = clean_model_text(data.get("reply", "")) or clean_model_text(raw)
    suggestions = data.get("suggested_replies", []) if isinstance(data.get("suggested_replies"), list) else []
    return {
        "character": profile,
        "reply": reply[:1200],
        "suggested_replies": [clean_model_text(item)[:80] for item in suggestions if clean_model_text(item)][:3],
    }


AVAILABLE_CHARACTERS = character_profiles(game)
print("[대화 가능한 미리 설정된 인물]")
for index, profile in enumerate(AVAILABLE_CHARACTERS, 1):
    print(f"{index}. {profile['name']} - {profile['description']}")

# 아래 이름을 위 목록에서 골라 바꾸세요.
CHAT_CHARACTER_NAME = AVAILABLE_CHARACTERS[0]["name"]
CHAT_MESSAGE = "모험을 하면서 가장 기억에 남는 순간은 무엇이었어?"
CHAT_RESULT = character_chat(game, CHAT_CHARACTER_NAME, CHAT_MESSAGE)
print(f"\n[{CHAT_CHARACTER_NAME}] {CHAT_RESULT['reply']}")
print("[다음 질문]", CHAT_RESULT["suggested_replies"])


In [ ]:
# 셀 12: 선택사항 - 내 목소리 WAV로 XTTS-v2 낭독을 테스트합니다.
# Flutter에서 녹음해 받은 PCM WAV 경로를 넣으면 같은 방식으로 합성됩니다.
SPEAKER_WAV_PATH = ""  # 예: /content/my_voice.wav
TTS_TEXT = game["state"]["scenes"][0]["text"]

if SPEAKER_WAV_PATH:
    from IPython.display import Audio, display
    audio_bytes = synthesize_with_my_voice(TTS_TEXT, SPEAKER_WAV_PATH)
    output_wav = Path("/content/test_v1_xtts_output.wav")
    output_wav.write_bytes(audio_bytes)
    print("XTTS 합성 완료:", output_wav)
    display(Audio(data=audio_bytes, rate=24000, autoplay=False))
else:
    print("SPEAKER_WAV_PATH가 비어 있어 XTTS 모델 로드를 건너뛰었습니다.")
    print("Flutter 녹음 파일 또는 3초 이상 PCM WAV의 경로를 넣으면 XTTS-v2로 내 목소리 낭독을 테스트합니다.")


In [ ]:
# 셀 13: 계획, 장면, 선택별 감정, 심리 해설을 Colab 로컬 JSON으로 저장합니다.
from datetime import datetime

output_path = Path("/content") / f"test_v1_story_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
output_path.write_text(json.dumps({
    "plan": game["plan"],
    "scenes": game["state"]["scenes"],
    "selected_choices": game["state"]["history"],
    "psych_report": globals().get("PSYCH_REPORT"),
    "ended": game["state"]["ended"],
}, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장 완료:", output_path)

# 필요하면 다음 두 줄의 주석을 풀어 파일을 내려받을 수 있습니다.
# from google.colab import files
# files.download(str(output_path))
